## Structured output
 
 Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.


#### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.6-27b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15'}}, client=<groq.resources.chat.completions.Completions object at 0x0000021C59692CF0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021C59693A10>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="Year in which the movie released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

In [6]:
model_with_structured_output=model.with_structured_output(Movie)
model_with_structured_output

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15'}}, client=<groq.resources.chat.completions.Completions object at 0x0000021C59692CF0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021C59693A10>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'Year in which the movie released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name': 'Mov

In [4]:
response=model.invoke("Tell me the details about Interstailer movie")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "Tell me the details about Interstailer movie"\n   - Key term: "Interstailer"\n   - I recognize this as likely a misspelling or confusion with "Interstellar" (the 2014 Christopher Nolan film)\n   - I should verify if "Interstailer" is an actual movie, but I\'m highly confident it\'s a typo for "Interstellar"\n\n2.  **Identify Core Subject:**\n   - Movie: *Interstellar* (2014)\n   - Director: Christopher Nolan\n   - Genre: Sci-fi/Adventure/Drama\n   - Key themes: Space exploration, time dilation, human survival, father-daughter relationship, love across dimensions\n   - Cast: Matthew McConaughey, Anne Hathaway, Jessica Chastain, Michael Caine, Matt Damon, etc.\n   - Plot summary: Earth is dying, astronauts travel through a wormhole near Saturn to find a new habitable planet for humanity\n   - Scientific aspects: Based on theoretical physics (Kip Thorne as executive producer/scientif

In [7]:
response=model_with_structured_output.invoke("Tell me the details about Interstailer movie")
response

Movie(title='Interstellar', year=2014, director='Christopher Nolan', rating=8.7)

#### Message output alongside parsed structure

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A Movie with Details"""
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="The year in which the movie released")
    director:str=Field(...,description="The director of the movie")
    rating:float=Field(...,description="The movie's rating out of 10")

In [9]:
model_with_structured_output=model.with_structured_output(Movie, include_raw=True)
response=model_with_structured_output.invoke("Provide the details about the movie PK")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:** The user is asking for details about the movie "PK".\n2.  **Identify Key Information Needed:** I need to provide details about the movie "PK". Based on the available function `Movie`, I need:\n   - title: "PK"\n   - year: Need to know the release year of the movie PK\n   - director: Need to know the director of PK\n   - rating: Need to know the rating (out of 10)\n3.  **Retrieve Knowledge (Internal):**\n   - Movie: PK (2014 Indian comedy-drama film)\n   - Director: Rajkumar Hirani\n   - Release Year: 2014\n   - Rating: Generally highly rated. On IMDb it\'s around 8.1/10. I\'ll use a reasonable rating like 8.1 or 8.2. Let\'s go with 8.1 as it\'s commonly cited.\n   - Check function parameters: title (string), year (integer), director (string), rating (number). All required.\n4.  **Construct Function Call:**\n   - title: "PK"\n   - year: 2014\n   - director: "

#### Nasted Structure

In [20]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None=Field(None, description="Budget in Indian Rupees")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie PK")
response

MovieDetails(title='PK', year=2014, cast=[Actor(name='Aamir Khan', role='PK'), Actor(name='Anushka Sharma', role='Jagat Janu'), Actor(name='Sushant Singh Rajput', role='Chaiwala'), Actor(name='Boman Irani', role='Pandit Chaturbhuj'), Actor(name='Anupam Kher', role='Gopal Singh')], genres=['Comedy', 'Drama', 'Science Fiction'], budget=None)

### TypedDict

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [21]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.6-27b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15'}}, client=<groq.resources.chat.completions.Completions object at 0x0000021C5C0C51D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021C5C0C5BD0>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [28]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie Conjurng")
response

{'director': 'James Wan',
 'rating': 7.5,
 'title': 'The Conjuring',
 'year': 2013}

In [31]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'The Avengers',
 'year': 2012}

### Dataclasses

A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")



In [43]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()


class ContactInfo(BaseModel):
    """Contact information for a person."""

    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


agent = create_agent(
    model=model,
    response_format=ContactInfo
)


result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})


contact = result["structured_response"]

print(contact)

name='John Doe' email='john@example.com' phone='(555) 123-4567'


#### using typedict

In [46]:
from typing_extensions import TypedDict
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()


class ContactInfo(TypedDict):
    """Contact information for a person."""

    name: str 
    email: str 
    phone: str


model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


agent = create_agent(
    model=model,
    response_format=ContactInfo
)


result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})


contact = result["structured_response"]

print(contact)

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


#### using dataclasses

In [47]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

@dataclass
class ContactInfo:
    """Contact information for a person."""

    name: str 
    email: str 
    phone: str


model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


agent = create_agent(
    model=model,
    response_format=ContactInfo
)


result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})


contact = result["structured_response"]

print(contact)

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')
